[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_2_Lagrange_Method_AR.ipynb) [![Launch JupyterLab](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hany73-max/PDE-Rover-Engine/main?urlpath=lab/tree/labs_ar/Lab_2_Lagrange_Method_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        else:
            import subprocess
            subprocess.run(['git', '-C', 'PDE-Rover-Engine', 'pull', '--ff-only'], check=False)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

# ‏🧭 اللاب 2: المعادلات شبه الخطية وطريقة لاجرانج لخطوط الانسياب

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المقرر:</b> المعادلات التفاضلية الجزئية وتطبيقاتها في ملاحة المركبات الفضائية

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المرجع في المذكرة:</b> الباب الأول، القسم 1.1.7 (ص 4–5) والقسم 1.3.1–1.3.2 (صفحات المذكرة 11–16 / صفحات PDF: 12–17)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏🗺️ <b>أين يقع هذا اللاب في خط عمل المشروع؟</b><br>
‏‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">خريطة الارتفاعات</code>‏ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longrightarrow}$</span>‏ ‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">كشف العوائق</code>‏ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longrightarrow}$</span>‏ ‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">نطاق الأمان</code>‏ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longrightarrow}$</span>‏ ‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">معادلة لابلاس φ(x,y)</code>‏ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longrightarrow}$</span>‏ <b>‏<code dir="rtl" style="direction:rtl; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">[اللاب 2: تتبع خط الانسياب descend()]</code>‏</b><br>
‏<b>الملف الحقيقي في المشروع:</b> <a href="../rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/planners.py</code>‏</a> — الدالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">descend(P, hazard, start, goal)</code>‏

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🧭 قاموس الربط: الرياضيات ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longleftrightarrow}$</span>‏ المعنى الفيزيائي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\longleftrightarrow}$</span>‏ كود المشروع

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏1. الرمز في مذكرة المحاضرة

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏2. المعنى الفيزيائي على أرض المريخ

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏3. الكود الفعلي في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">planners.py</code>‏

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${P(x,y,z)\,p + Q(x,y,z)\,q = R(x,y,z)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>معادلة لاجرانج شبه الخطية:</b> المشتقتان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p, q}$</span>‏ من الدرجة الأولى

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\phi_x \frac{\partial\psi}{\partial x} + \phi_y \frac{\partial\psi}{\partial y} = 0}$</span>‏ (معادلة دالة الانسياب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\psi}$</span>‏)

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>معادلات لاجرانج المساعدة:</b> معادلات المسار الذي يسلكه الجسيم خطوة بخطوة

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{dx}{-\phi_x} = \frac{dy}{-\phi_y}}$</span>‏ (السير مع أشد انحدار)

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${u(x,y,z) = a, \quad v(x,y,z) = b}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏ثابتان يبقيان محفوظين طوال حركة المركبة على نفس المسار

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏منحنى مسار المركبة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">path = [(x, y), ...]</code>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\Phi(u, v) = 0}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>الحل العام:</b> يجمع كل المسارات الممكنة بدالة اختيارية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\Phi}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏عائلة خطوط الانسياب الكاملة من البداية للهدف

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏1. الفكرة ببساطة: كيف تتحول معادلة تفاضلية جزئية إلى مسار قيادة؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏بعد أن يحسب حاسوب المركبة حقل الجهد التوافقي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi(x, y)}$</span>‏ (حيث البداية عند ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi=1}$</span>‏ والهدف عند ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi=0}$</span>‏)، كيف تقود المركبة عجلاتها؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏المركبة تتجه في كل لحظة باتجاه <b>أشد انحدار للأسفل (Steepest Descent)</b>:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\mathbf{v} = (u, v) = -\nabla\phi = \left(-\frac{\partial\phi}{\partial x},\; -\frac{\partial\phi}{\partial y}\right)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذا أردنا إيجاد <b>دالة الانسياب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\psi(x, y)}$</span>‏</b> التي تبقى قيمتها ثابتة على طول مسار المركبة (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${d\psi = 0}$</span>‏ على المسار)، فإن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\psi}$</span>‏ تحقق معادلة لاجرانج الخطية من الرتبة الأولى:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$u \frac{\partial\psi}{\partial x} + v \frac{\partial\psi}{\partial y} = 0 \qquad \text{أي أن:} \qquad P = u,\; Q = v,\; R = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏وعندما نكتب <b>معادلات لاجرانج المساعدة</b> لهذه المعادلة نحصل على:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{dx}{u} = \frac{dy}{v} = \frac{d\psi}{0} \implies \psi(x,y) = \text{ثابت على طول مسار المركبة!}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذن: <b>المنحنيات المميزة لطريقة لاجرانج (Characteristic Curves) هي بالضبط المسارات الهندسية التي تسير عليها عجلات المركبة في دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">descend()</code>‏!</b>

</div>

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab2

%matplotlib inline
print("تم تهيئة بيئة اللاب 2 بنجاح!")


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏2. قاعدة العمل والاستراتيجيات الثلاث لحل معادلات لاجرانج في المذكرة (PDF ص 12–17)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🎯 كيف أميّز معادلة لاجرانج في الامتحان؟ (صفحة 4 و 11 بالمذكرة / PDF ص 5 و 12)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏انظر إلى المشتقتين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q}$</span>‏: إذا كانتا مرفوعتين <b>للأس 1 فقط</b> ولا يوجد ضرب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p \cdot q}$</span>‏، فالمعادلة على الصورة القياسية للاجرانج:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$P(x, y, z)\,p + Q(x, y, z)\,q = R(x, y, z)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏وتُحل دائماً بكتابة <b>معادلات لاجرانج المساعدة (Lagrange's Auxiliary Equations)</b>:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏ثم نوجد <b>ثابتين مستقلين</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u(x, y, z) = a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${v(x, y, z) = b}$</span>‏، ويكون الحل العام هو:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\Phi(u, v) = 0 \qquad (\text{حيث } \Phi \text{ دالة اختيارية})$$

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🧠 الاستراتيجيات الثلاث في المذكرة لإيجاد الثابتين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏ (PDF ص 13–16)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في الامتحان، عند النظر إلى الكسور ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}}$</span>‏، ستواجه إحدى <b>الحالات الثلاث التالية بالضبط</b> (وقد وردت جميعها في أمثلة المذكرة من Ex.1 إلى Ex.4):

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🔹 الاستراتيجية 1: الفصل المباشر بين كسرين (Ex.1 & Ex.2 — ص 12–13 بالمذكرة / PDF ص 13–14)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عندما يحتوي كسران على متغيرين فقط، نضرب طرفين في وسطين ونكامل مباشرة!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 1 (PDF ص 13):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${2p - 3q = 2x}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ المعاملات: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${P = 2,\; Q = -3,\; R = 2x \implies \frac{dx}{2} = \frac{dy}{-3} = \frac{dz}{2x}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الكسر الأول والثاني:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dx}{2} = \frac{dy}{-3} \implies -3\,dx - 2\,dy = 0 \xrightarrow{\int} -3x - 2y = a}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الكسر الأول والثالث:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dx}{2} = \frac{dz}{2x} \implies x\,dx - dz = 0 \xrightarrow{\int} \frac{x^2}{2} - z = b}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الحل العام:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\Phi\left(-3x - 2y,\; \frac{x^2}{2} - z\right) = 0}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 2 (PDF ص 13–14):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\left(\frac{y^2 z}{x}\right)p + xzq = y^2}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ المعادلات المساعدة: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dx}{y^2 z / x} = \frac{dy}{xz} = \frac{dz}{y^2}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الكسر الأول والثاني:</b> نختصر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ من المقامين ونضرب طرفين في وسطين: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x^2\,dx - y^2\,dy = 0 \implies x^3 - y^3 = a_1}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الكسر الأول والثالث:</b> نختصر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y^2}$</span>‏ من المقامين: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x\,dx - z\,dz = 0 \implies x^2 - z^2 = b_1}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الحل العام:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\Phi(x^3 - y^3,\; x^2 - z^2) = 0}$</span>‏.

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🔹 الاستراتيجية 2: استخدام الثابت الأول ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ بالتعويض لحل الكسر التالي ثم إرجاعه! (Ex.3 — ص 13–14 بالمذكرة / PDF ص 14–15)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏ماذا لو كان الكسر الأخير يحتوي على متغيرات إضافية تمنع فصل المتغيرات مباشرة؟ <b>نستخدم الثوابت التي أوجدناها للتو لتبديل المتغيرات، وبعد التكامل نُرجع الثوابت إلى قيمها الأصلية!</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 3 (PDF ص 14–15):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x\frac{\partial z}{\partial x} + y\frac{\partial z}{\partial y} + t\frac{\partial z}{\partial t} = xyt}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ المعادلات المساعدة: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dx}{x} = \frac{dy}{y} = \frac{dt}{t} = \frac{dz}{xyt}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الكسر 1 و 2:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dx}{x} = \frac{dy}{y} \implies \ln x - \ln y = \ln a \implies \frac{x}{y} = a}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الكسر 2 و 3:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dy}{y} = \frac{dt}{t} \implies \ln y - \ln t = \ln b \implies \frac{y}{t} = b}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الكسر 2 و 4:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dy}{y} = \frac{dz}{xyt} \implies xt\,dy - dz = 0}$</span>‏. <i>(لا يمكن تكاملها مباشرة لوجود ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${t}$</span>‏ مع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${dy}$</span>‏!)</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الحيلة:</b> من الثابتين السابقين لدينا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x = a y}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${t = \frac{y}{b} \implies xt = \frac{a}{b}y^2}$</span>‏. نعوض بها في المعادلة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{a}{b}y^2\,dy - dz = 0 \xrightarrow{\text{نكامل الآن بسهولة!}} \frac{a}{3b}y^3 - z = c$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>خطوة لا تنسها أبداً:</b> قبل كتابة الحل العام، أرجع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = \frac{x}{y}}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b = \frac{y}{t}}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{(x/y)}{3(y/t)}y^3 - z = c \implies \frac{1}{3}xyt - z = c \implies \Phi\left(\frac{x}{y},\; \frac{y}{t},\; \frac{1}{3}xyt - z\right) = 0$$

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏🔹 الاستراتيجية 3: قاعدة تساوي الكسور / المضروبات (القاعدة المؤطرة في ص 14 و Ex.4 ص 15 / PDF ص 15–16)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏📌 <b>القاعدة الذهبية في المذكرة (PDF ص 15):</b><br>
‏<i>"لأي كسور متساوية، <b>إذا كان مجموع المقامات يساوي صفراً، فإن مجموع البسوط يساوي صفراً أيضاً!</b>"</i><br>
‏وبصورة أعم، إذا ضربنا البسوط والمقامات في معاملات (مضروبات) ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(l, m, n)}$</span>‏ بحيث يصبح مجموع المقامات الجديد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${lP + mQ + nR = 0}$</span>‏، فإن البسط الجديد يساوي صفراً أيضاً:<br>
‏$$l\,dx + m\,dy + n\,dz = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 4 (PDF ص 16 — سؤال امتحانات كلاسيكي هام جداً!):</b> حل المعادلة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(y - z)p + (z - x)q = x - y$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الخطوة 1:</b> المعادلات المساعدة هي:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{dx}{y - z} = \frac{dy}{z - x} = \frac{dz}{x - y}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<i>(لاحظ أن كل مقام يحتوي على متغيرين مختلطين، فلا يمكن فصل أي كسرين مباشرة!)</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الخطوة 2 (إيجاد الثابت الأول بجمع المقامات مباشرة):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏اجمع المقامات الثلاثة: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(y - z) + (z - x) + (x - y) = 0}$</span>‏!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏بما أن مجموع المقامات ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0}$</span>‏، إذن مجموع البسوط ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$dx + dy + dz = 0 \xrightarrow{\text{بتكامل الطرفين}} x + y + z = a$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الخطوة 3 (إيجاد الثابت الثاني بضرب الكسور في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x, y, z}$</span>‏ على الترتيب):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏اضرب الكسر الأول في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ والثاني في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ والثالث في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ واجمع المقامات:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$x(y - z) + y(z - x) + z(x - y) = xy - xz + yz - xy + xz - yz = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏مجموع المقامات الجديد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0}$</span>‏، إذن مجموع البسوط الجديد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$x\,dx + y\,dy + z\,dz = 0 \xrightarrow{\text{بتكامل الطرفين}} \frac{x^2}{2} + \frac{y^2}{2} + \frac{z^2}{2} = b$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الخطوة 4 (الحل العام):</b>

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\Phi\left(x + y + z,\; \frac{x^2}{2} + \frac{y^2}{2} + \frac{z^2}{2}\right) = 0$$

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏3. ✍️ تحدي الورقة والقلم 1 (التمرين 5 — صفحة 16 بالمذكرة / PDF ص 17)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة (معادلة أويلر):</b> أوجد الثابتين المستقلين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u(x,y)}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${v(y,z)}$</span>‏ للمعادلة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$x p + y q = z$$

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 1:</b> اكتب المعادلات المساعدة

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏هنا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${P = x, \; Q = y, \; R = z}$</span>‏، إذن:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{dx}{x} = \frac{dy}{y} = \frac{dz}{z}$$

</div>

</details>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 2:</b> كامل الكسر الأول مع الثاني، ثم الثاني مع الثالث

</summary>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\int \frac{dx}{x} = \int \frac{dy}{y} \implies \ln|x| - \ln|y| = \ln|a| \implies \ln\left|\frac{x}{y}\right| = \ln|a|$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏وبالمثل للكسر الثاني والثالث مع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏.

</div>

</details>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏أدخل صيغتي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${v}$</span>‏ في الخلية التالية:

</div>

</div>


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ### (≈ 2 lines of code)
# اكتب الثابتين المستقلين u و v المستنتجين من التكامل (مثال: "x/y" و "y/z"):
student_u = None
student_v = None
### END CODE HERE ###

test_lab2.check_paper_challenge_1(student_u, student_v)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏4. 🚀 تحدي الربط بالمشروع: استنتاج معادلة خط انسياب المركبة

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لنفترض أن حقل الجهد بالقرب من الهدف يعطى بالدالة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\phi(x, y) = x^2 + 2y^2$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏مركبة المريخ تتحرك باتجاه أشد انحدار للأسفل بسرعة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$u = -\frac{\partial\phi}{\partial x} = -2x, \qquad v = -\frac{\partial\phi}{\partial y} = -4y$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذن <b>دالة الانسياب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\psi(x, y)}$</span>‏</b> التي تبقى ثابتة على مسار المركبة تحقق معادلة لاجرانج:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$-2x \frac{\partial\psi}{\partial x} - 4y \frac{\partial\psi}{\partial y} = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المطلوب على ورقتك:</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏استخدم معادلة لاجرانج المساعدة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dx}{-2x} = \frac{dy}{-4y}}$</span>‏ لإيجاد معادلة منحنى مسار المركبة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ بدلالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ والثابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${k}$</span>‏.

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 1:</b> بسّط المقامات وفصل المتغيرات

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏بقسمة المقامين على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-2}$</span>‏ نحصل على:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{dx}{x} = \frac{dy}{2y} \implies 2\frac{dx}{x} = \frac{dy}{y}$$

</div>

</details>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 2:</b> كامل الطرفين وخذ الدالة الأسية

</summary>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$2\ln|x| + \ln|k| = \ln|y| \implies \ln(k x^2) = \ln|y| \implies y = k x^2$$

</div>

</details>

</div>


In [ ]:
# خلية تقييم: معادلة مسار المركبة
### START CODE HERE ### (≈ 1 line of code)
# اكتب معادلة مسار الانسياب بدلالة x و y و k (مثال: "y = k*x^2"):
rover_streamline = None
### END CODE HERE ###

test_lab2.check_paper_challenge_2(rover_streamline)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏5. 💻 من الورقة إلى الكود: متتبع خطوط الانسياب العددي

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏على تضاريس المريخ الحقيقية لا نملك صيغة جبرية بسيطة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi(x,y)}$</span>‏، بل لدينا شبكة أرقام. لذلك يتتبع الحاسوب معادلة لاجرانج المساعدة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{dx}{dt} = u(x,y), \qquad \frac{dy}{dt} = v(x,y)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏بأخذ خطوات زمنية صغيرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\Delta t}$</span>‏ (طريقة أويلر الأمامية أو طريقة رونج-كوتا):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(x_{\text{new}}, y_{\text{new}}) = (x, y) + \Delta t \cdot (u, v)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏وهي الفكرة ذاتها التي تنفذها دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">descend()</code>‏ على الخلايا الثماني المجاورة في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">planners.py</code>‏!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏أكمل الدالة أدناه لتتبع مسار المركبة عددياً ومقارنته بحلك الورقي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y = 2x^2}$</span>‏ (عند الانطلاق من النقطة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(1, 2)}$</span>‏ حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${k = y_0/x_0^2 = 2}$</span>‏):

</div>

</div>


In [ ]:
# دالة تقييم برمجية: trace_streamline_rk4
def trace_streamline_rk4(velocity_fn, start=(1.0, 2.0), dt=0.01, steps=50):
    """تتبع خط الانسياب (المنحنى المميز للاجرانج) عددياً بخطوة dt."""
    trajectory = [start]
    curr = np.array(start, dtype=float)
    
    ### START CODE HERE ### (≈ 4-5 lines of code)
    # كرر بعدد steps:
    # 1. احسب مركبتي السرعة: vx, vy = velocity_fn(curr[0], curr[1])
    # 2. حدث الموقع: curr = curr + dt * np.array([vx, vy], dtype=float)
    # 3. أضف الموقع الجديد: trajectory.append(tuple(curr))
    pass
    ### END CODE HERE ###
    
    return np.array(trajectory)

test_lab2.test_rk4_streamline_tracer(trace_streamline_rk4)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏6. 🛰️ الرؤية البصرية: تطابق حل لاجرانج الورقي مع المتتبع العددي

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏شغّل الخلية التالية لترى كيف ينطبق المسار العددي (الخط الأحمر) تماماً فوق المنحنى التحليلي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y = 2x^2}$</span>‏ (الخط المتقطع الأزرق) الذي استنتجته بطريقة لاجرانج على الورق، وكيف يتعامد دائماً مع خطوط تساوي الجهد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi = \text{const}}$</span>‏:

</div>

</div>


In [ ]:
def sample_velocity(x, y):
    return -2.0 * x, -4.0 * y

traj = trace_streamline_rk4(sample_velocity, start=(1.0, 2.0), dt=0.01, steps=120)

x_vals = np.linspace(-2, 2, 25)
y_vals = np.linspace(-0.5, 2.5, 25)
X, Y = np.meshgrid(x_vals, y_vals)
PHI = X**2 + 2 * Y**2
U, V = sample_velocity(X, Y)

x_exact = np.linspace(0.05, 1.0, 100)
y_exact = 2.0 * (x_exact ** 2)

plt.figure(figsize=(8.5, 6))
plt.contour(X, Y, PHI, levels=12, cmap='viridis', alpha=0.5)
plt.streamplot(X, Y, U, V, color='lightgray', density=0.9)
plt.plot(x_exact, y_exact, 'b--', linewidth=3.5, label=r"Paper Solution (Lagrange): $y = 2x^2$")
plt.plot(traj[:, 0], traj[:, 1], 'r-', linewidth=2.2, label="Numerical Tracer (Code)")
plt.plot(1.0, 2.0, 'go', markersize=9, label="Start (1, 2)")
plt.plot(0.0, 0.0, 'y*', markersize=13, label="Goal (0, 0)")
plt.title("Lagrange Characteristic Curve vs. Numerical Streamline Tracer", fontsize=12)
plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()

print("📌 ملخص اللاب 2:")
print("  1) المعادلة شبه الخطية P*p + Q*q = R تُحل بمعادلات لاجرانج المساعدة dx/P = dy/Q = dz/R.")
print("  2) المنحنيات المميزة للاجرانج هي نفسها خطوط انسياب حركة المركبة في حقل الجهد.")
print("  3) دالة descend() في planners.py هي التطبيق الشبكي المباشر لتتبع هذه الخطوط من البداية إلى الهدف.")
